# Lab: OCR + RAG Chatbot (RapidOCR + Gemini)



### Step 1: Install Required Libraries

In [22]:
# rapidocr        -> the OCR engine (runs fully on CPU via onnxruntime)
# onnxruntime      -> the backend RapidOCR uses to run its models
# google-genai     -> the official Gemini API SDK (chat + embeddings)
# requests, numpy  -> downloading files & doing the similarity math
# fitz (pymupdf)   -> Reading pdfs
!pip install rapidocr onnxruntime google-genai requests numpy pymupdf --quiet

### Step 2: Import Libraries

In [23]:
# Standard library
import os
import json

# For downloading the sample invoices
import requests
# for doing vector math
import numpy as np
# For reading pdfs
import fitz

# RapidOCR -- our OCR engine
from rapidocr import RapidOCR

# Gemini SDK -- used both for embeddings (retrieval) and chat (generation)
from google import genai

### Step 3: Set Up the Gemini API Key

In [24]:
# Paste your key (get one for free at https://aistudio.google.com/apikey)
GEMINI_API_KEY = "YOUR GEMINI API KEY"

# Create one Gemini client we'll reuse for both embeddings and chat generation
client = genai.Client(api_key=GEMINI_API_KEY)

# Model names
CHAT_MODEL = "gemini-2.5-flash"
EMBED_MODEL = "gemini-embedding-001"

print("Gemini client ready.")

Gemini client ready.


### Step 4: Download Sample Invoice Images from GitHub


In [25]:
# Donwload Scanned PDF

PDF_URL = "https://raw.githubusercontent.com/Azure/azure-sdk-for-python/master/sdk/formrecognizer/azure-ai-formrecognizer/tests/sample_forms/forms/multipage_invoice1.pdf"

os.makedirs("invoices", exist_ok=True)
PDF_PATH = os.path.join("invoices", "multipage_invoice1.pdf")

response = requests.get(PDF_URL)
response.raise_for_status()
with open(PDF_PATH, "wb") as f:
    f.write(response.content)

print(f"Downloaded the scanned invoice PDF to '{PDF_PATH}' ({len(response.content)/1024:.1f} KB)")

Downloaded the scanned invoice PDF to 'invoices/multipage_invoice1.pdf' (106.4 KB)


In [26]:
# RapidOCR reads images, not PDFs directly, so we render each page to a PNG first.
# This is exactly what you'd do for a real scanned PDF, which has no text layer at all --
# we never look at the PDF's embedded text, only the rendered pixels.

doc = fitz.open(PDF_PATH)
invoice_paths = {}

for page_number, page in enumerate(doc, start=1):
    pix = page.get_pixmap(dpi=200)  # 200 DPI is enough detail for OCR without being slow. Higher DPI = More Pixels = More Time and computation
    filename = f"multipage_invoice1_page{page_number}.png"
    path = os.path.join("invoices", filename)
    pix.save(path)
    invoice_paths[filename] = path
    print(f"Rendered page {page_number} -> {filename}")

doc.close()

Rendered page 1 -> multipage_invoice1_page1.png
Rendered page 2 -> multipage_invoice1_page2.png
Rendered page 3 -> multipage_invoice1_page3.png


### Step 5: Run OCR on Each Invoice (RapidOCR)


In [27]:
# Initialize the OCR engine once -- this loads the detection, classification, and recognition models
engine = RapidOCR()

ocr_results = {}
for filename, path in invoice_paths.items():
    result = engine(path)
    ocr_results[filename] = result
    num_boxes = len(result.txts) if result.txts else 0
    print(f"{filename}: found {num_boxes} text boxes in {result.elapse:.2f}s")

[INFO] 2026-07-28 09:31:47,127 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-28 09:31:47,213 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-28 09:31:47,216 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-28 09:31:47,316 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-28 09:31:47,331 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-28 09:31:47,333 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-28 09:31:47,448 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-28 09:31:47,599 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

multipage_invoice1_page1.png: found 40 text boxes in 11.32s


[WARNING] 2026-07-28 09:32:06,841 [RapidOCR] main.py:132: The text detection result is empty


multipage_invoice1_page2.png: found 0 text boxes in 0.00s
multipage_invoice1_page3.png: found 40 text boxes in 9.53s


### Step 6: Convert OCR Output into Structured Text

In [28]:
# Build a dictionary of {filename: structured_text} -- this is our tiny document store
document_texts = {}
for filename, result in ocr_results.items():
    if not result.txts:
        print(f"Skipping {filename} -- no text detected (blank page)")
        continue
    document_texts[filename] = result.to_markdown()

# Peek at one example to see the structure preserved
print(document_texts.keys())

Skipping multipage_invoice1_page2.png -- no text detected (blank page)
dict_keys(['multipage_invoice1_page1.png', 'multipage_invoice1_page3.png'])


### Step 7: Embed Each Document (Building the Knowledge Base)


In [29]:
def embed_text(text):
    """Get a Gemini embedding vector for a piece of text."""
    response = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return np.array(response.embeddings[0].values)

# Build the knowledge base: one entry per document, with its text and embedding
knowledge_base = []
for filename, text in document_texts.items():
    knowledge_base.append({
        "source": filename,
        "text": text,
        "embedding": embed_text(text),
    })

print(f"Knowledge base built with {len(knowledge_base)} documents.")

Knowledge base built with 2 documents.


### Step 8: Define the Retrieval Function (Cosine Similarity)


In [30]:
def cosine_similarity(a, b):
    # Standard cosine similarity: dot product over the product of magnitudes
    magnitude_a = np.sqrt(np.sum(a**2))
    magnitude_b = np.sqrt(np.sum(b**2))
    return np.dot(a, b) / (magnitude_a * magnitude_b)

def retrieve(query, top_k=2):
    """Return the top_k most relevant documents for the query, most relevant first."""
    query_embedding = embed_text(query)

    scored = []
    for doc in knowledge_base:
        score = cosine_similarity(query_embedding, doc["embedding"])
        scored.append({**doc, "score": float(score)})

    scored.sort(key=lambda d: d["score"], reverse=True)
    return scored[:top_k]

### Step 9: Build the RAG Pipeline (Retrieve + Ask Gemini)

In [31]:
def rag_answer(query, top_k=1):
    retrieved_docs = retrieve(query, top_k=top_k) #top_k decides number of documents to fetch

    # Label each document so Gemini can cite exactly which one it used
    labeled_context = "\n\n".join(
        f"[Source: {doc['source']}]\n{doc['text']}" for doc in retrieved_docs
    )

    prompt = f"""You are an assistant that answers questions about invoices/receipts using ONLY the context below.
Every fact you state must be tagged with its source, like [Source: filename].
If the answer isn't in the context, say "Not found in the provided documents."

Context:
{labeled_context}

Question: {query}
Answer:"""

    response = client.models.generate_content(model=CHAT_MODEL, contents=prompt)
    return response.text, retrieved_docs

### Step 10: Ask a Question

In [32]:
query = "What is the total on Company A's invoice, and who is it addressed to?"

answer, retrieved_docs = rag_answer(query)

print("--- DOCUMENTS RETRIEVED ---")
for doc in retrieved_docs:
    print(f"{doc['source']}  (similarity: {doc['score']:.3f})")

print("\n--- ANSWER ---")
print(answer)

--- DOCUMENTS RETRIEVED ---
multipage_invoice1_page1.png  (similarity: 0.768)

--- ANSWER ---
The total on Company A's invoice is 430.00 [Source: multipage_invoice1_page1.png]. It is addressed to Bilbo Baggins [Source: multipage_invoice1_page1.png].


In [33]:
print("\n--- EXPLAINABILITY ---")
for doc in retrieved_docs:
    # Check if the citation tag is actually present in Gemini's answer
    was_used = f"[Source: {doc['source']}]" in answer

    status = "USED in answer" if was_used else "retrieved but NOT used"

    print(f"\nDocument: \"{doc['source']}\"")
    print(f"similarity: {doc['score']:.3f} | {status}")

    # Ask Gemini why this document is relevant, using the actual text that was retrieved
    explain_prompt = f"""
In 3-4 short lines, explain why the document below is relevant to the question.
Be specific -- mention the actual numbers or fields that connect to the question.
Do not repeat the question. Do not add extra commentary.

Question: {query}

Document source: {doc['source']}
Document content: {doc['text']}
"""
    explanation_response = client.models.generate_content(model=CHAT_MODEL, contents=explain_prompt)
    print(f"Why: {explanation_response.text.strip()}")


--- EXPLAINABILITY ---

Document: "multipage_invoice1_page1.png"
similarity: 0.768 | USED in answer
Why: This document is an invoice from "Company A". It specifies the recipient as "Bilbo Baggins" under "Invoice For". The total amount on this invoice is clearly stated as "430.00" at the bottom.


### Step 11: Simple Interactive Chatbot

In [ ]:
while True:
    question = input("Ask about your invoices (or 'exit'): ").strip()
    if question.lower().strip() == "exit":
        print("Goodbye!")
        break

    answer, retrieved_docs = rag_answer(question)
    sources = ", ".join(doc["source"] for doc in retrieved_docs)
    print(f"\n[Retrieved: {sources}]")
    print(answer)
    print()

Ask about your invoices (or 'exit'): Tell me the names of all recipients of invoices. 

[Retrieved: multipage_invoice1_page1.png]
The recipient of the invoice is Bilbo Baggins [Source: multipage_invoice1_page1.png].

Ask about your invoices (or 'exit'): exit
Goodbye!
